# RDO filter -- hardware selection vs software golden

Validates the RDO combiner IP end-to-end on 13 test sequences.

**What it checks:** for every full 256x256 restoration unit (RU) in each frame,
does the FPGA pick the same NONE/PC/NS winner as the bit-exact software model?

**What it runs per sequence:**
1. Train per-RU NS taps from the pre-LR / HR frame pair.
2. Full-frame **PC** filter on hardware.
3. Full-frame **NS** filter on hardware.
4. Feed (HR, PRELR, PC-HW, NS-HW) into the RDO IP, one RU at a time.
5. Read the 3 selection bitmaps back over AXI-Lite.
6. Compare against the SW golden's MSE-argmin (tie-break NONE > PC > NS).

**Success criterion:** 100.00% per-RU agreement, every sequence.

Secondary reporting: mean PSNR of the RDO-blended output vs HR.


In [ ]:
from pynq import Overlay, allocate
import pynq, pynq.ps, pynq.lib.video
import numpy as np
import os, glob, time, gc

# ---- Paths (adjust BASE_RDO / DATASETS / PC_NPZ if the board layout differs) ----
BASE_RDO = '/home/xilinx/jupyter_notebooks/rdo_filter'
DATASETS = '/home/xilinx/jupyter_notebooks/datasets'
PC_NPZ   = '/home/xilinx/jupyter_notebooks/pc_filter/pc_qp_group_1.npz'

ol      = Overlay(f'{BASE_RDO}/rdo_filter_bd_wrapper.bit')
rdo     = ol.rdo_filter_axi_0
ns_flt  = ol.ns_filter_0
pc_flt  = ol.pc_filter_0
dma_hr  = ol.axi_dma_0    # HR       -> RDO
dma_no  = ol.axi_dma_1    # NONE     -> RDO
dma_pc  = ol.axi_dma_2    # PC-HW   -> RDO
dma_ns  = ol.axi_dma_3    # NS-HW   -> RDO
dma_nsf = ol.axi_dma_4    # NS filter (mm2s + s2mm)
dma_pcf = ol.axi_dma_5    # PC filter (mm2s + s2mm)

# Overlay-load side-effect: DMAs are already 'running' -- suppress PYNQ's
# start() sanity check so the first transfer isn't refused.
for d in [dma_hr, dma_no, dma_pc, dma_ns, dma_nsf, dma_pcf]:
    if getattr(d, 'sendchannel', None) is not None:
        d.sendchannel._first_transfer = True
    if getattr(d, 'recvchannel', None) is not None:
        d.recvchannel._first_transfer = True

# ---- RDO register offsets (from rdo_filter_axi.v) ----
CTRL, LAMBDA          = 0x00, 0x04
R_NONE, R_PC, R_NS    = 0x08, 0x0C, 0x10
STATUS                = 0x14
SEL_NONE_LO, SEL_NONE_HI = 0x18, 0x1C
SEL_PC_LO,   SEL_PC_HI   = 0x20, 0x24
SEL_NS_LO,   SEL_NS_HI   = 0x28, 0x2C
CTRL_CLEAR, CTRL_SOF     = 0x1, 0x4

# Rate cost knobs. Zero means 'pick the argmin(MSE) directly'.
LAMBDA_VAL   = 0
R_NONE_VAL, R_PC_VAL, R_NS_VAL = 0, 0, 0
rdo.write(LAMBDA, LAMBDA_VAL)
rdo.write(R_NONE, R_NONE_VAL)
rdo.write(R_PC,   R_PC_VAL)
rdo.write(R_NS,   R_NS_VAL)

# ---- Geometry ----
RU_SIZE = 256
N_RU_PX = RU_SIZE * RU_SIZE       # 65536

# ---- Persistent CMA buffers (feedback_pynq_cma_buffer_reuse) ---------------
# Allocate ONCE, reuse via slice views. Per-frame allocate/freebuffer churn
# fragments CMA and leaves dangling DMA descriptors -> exit-time segfault.
MAX_W, MAX_H = 1920, 1920

IN_BUF     = allocate(shape=(MAX_H * MAX_W,),             dtype=np.uint32)
OUT_NS_BUF = allocate(shape=((MAX_H - 6) * MAX_W,),       dtype=np.uint32)
OUT_PC_BUF = allocate(shape=((MAX_H - 6) * (MAX_W - 6),), dtype=np.uint32)

RU_HR = allocate(shape=(N_RU_PX,), dtype=np.uint32)
RU_NO = allocate(shape=(N_RU_PX,), dtype=np.uint32)
RU_PC = allocate(shape=(N_RU_PX,), dtype=np.uint32)
RU_NS = allocate(shape=(N_RU_PX,), dtype=np.uint32)

total_mb = sum(b.nbytes for b in
               [IN_BUF, OUT_NS_BUF, OUT_PC_BUF, RU_HR, RU_NO, RU_PC, RU_NS]) / 1e6
print(f"overlay loaded, persistent CMA buffers = {total_mb:.1f} MB")


## Sequences

Same 13-sequence panel as the PC/NS notebooks (10 landscape + 3 portrait).
`find_representative` picks the first `(range, frame, qp)` triple in each
sequence dir that has a matching HR frame.


In [ ]:
RESOLUTIONS = {
    'CrowdRun_1920x1080p50':               (1920, 1080),
    'Motorcycle_1920x1080_30fps_8bit':     (1920, 1080),
    'MountainBike_1920x1080_30fps_8bit':   (1920, 1080),
    'OldTownCross_1920x1080p50':           (1920, 1080),
    'PedestrianArea_1920x1080p25':         (1920, 1080),
    'Riverbed_1920x1080p25':               (1920, 1080),
    'RushFieldCuts_1920x1080_2997':        (1920, 1080),
    'TreesAndGrass_1920_1080_30fps_8bit':  (1920, 1080),
    'Vertical_Carnaby_1080x1920_5994':     (1080, 1920),
    'Vertical_bees_1080x1920_2997':        (1080, 1920),
    'WalkingInStreet_1920x1080_30fps':     (1080, 1920),
    'WorldCup_1920x1080_30p':              (1920, 1080),
    'WorldCup_far_1920x1080_30p':          (1920, 1080),
}

def load_raw_y(path, W, H):
    return np.fromfile(path, dtype=np.uint8).reshape(H, W)


def find_representative(seq_dir, qp=None):
    """Pick one (prelr, hr) pair for this sequence, optionally matching a qp.

    Returns (prelr_path, hr_path, rng, frame_id, qp) or (None,)*5.
    """
    prelr_root = os.path.join(seq_dir, 'TEST_PRELR')
    hr_root    = os.path.join(seq_dir, 'TEST_HR')
    if not (os.path.isdir(prelr_root) and os.path.isdir(hr_root)):
        return (None,) * 5
    for rng in sorted(os.listdir(prelr_root)):
        prelr_dir = os.path.join(prelr_root, rng)
        if not os.path.isdir(prelr_dir):
            continue
        for f in sorted(os.listdir(prelr_dir)):
            if not f.endswith('_prelr.yuv'):
                continue
            parts    = f.replace('_prelr.yuv', '').split('_')
            frame_id = parts[1]
            file_qp  = parts[2].replace('qp', '')
            if qp is not None and file_qp != qp:
                continue
            hr_path  = os.path.join(hr_root, f'frame_{frame_id}.y')
            if os.path.exists(hr_path):
                return os.path.join(prelr_dir, f), hr_path, rng, frame_id, file_qp
    return (None,) * 5


# QP sweep -- matches the QPs pc_qp_group_1.npz was trained for (group_1: 160/170/180).
# Running PC out of its trained QP range (e.g. QP 110) makes PC underperform
# and biases RDO toward NS/NONE unfairly. See memory/project_avm_bdrate_results.
QPS = ['160', '170', '180']


# TEST split used in the Python RDO pipeline (paper Fig 4 subset).
# Do NOT add training sequences (Motorcycle, MountainBike, TreesAndGrass,
# WalkingInStreet, WorldCup) or QA-only verticals here -- mixing them into
# evaluation invalidates the comparison to the Python BD-Rate results.
TEST_SEQUENCES = [
    'CrowdRun_1920x1080p50',
    'OldTownCross_1920x1080p50',
    'PedestrianArea_1920x1080p25',
    'Riverbed_1920x1080p25',
    'RushFieldCuts_1920x1080_2997',
]

available = set(d for d in os.listdir(DATASETS)
                if os.path.isdir(os.path.join(DATASETS, d)))
sequences = [s for s in TEST_SEQUENCES if s in available]
missing   = [s for s in TEST_SEQUENCES if s not in available]

print(f"evaluating on {len(sequences)} / {len(TEST_SEQUENCES)} test sequences")
for s in sequences:
    print(f"  ok      {s}")
for s in missing:
    print(f"  MISSING {s}  (skipped)")


## Software golden and HW helpers

- `sw_select_tile`: bit-exact port of `cost_compare.v`. Cost = SUM((cand-hr)**2)
  + lambda * R. Tie-break NONE > PC > NS.
- `train_ns_taps`: same Wiener-KKT + coordinate-descent trainer as
  `ns_golden.py`, inlined so this notebook is self-contained.
- `run_pc_hw`, `run_ns_hw`: full-frame filter runs. Interior is filtered,
  3-pixel border copied from pre-LR (matches what each filter IP produces).
- `run_rdo_frame`: streams every full 256x256 RU through the 4-way RDO input,
  reads the 3 selection bitmaps, returns per-RU winners.
- `bits_to_winners`: unpack a 64-bit selection mask into a list of 0/1/2.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view
from collections import Counter

# ============================================================================
# SW golden -- mirrors cost_compare.v exactly
# ============================================================================
def sw_select_tile(hr_t, none_t, pc_t, ns_t,
                   lam=LAMBDA_VAL, rn=R_NONE_VAL, rp=R_PC_VAL, rs=R_NS_VAL):
    d0 = none_t.astype(np.int32) - hr_t.astype(np.int32)
    d1 = pc_t  .astype(np.int32) - hr_t.astype(np.int32)
    d2 = ns_t  .astype(np.int32) - hr_t.astype(np.int32)
    m0 = int(np.sum(d0 * d0))
    m1 = int(np.sum(d1 * d1))
    m2 = int(np.sum(d2 * d2))
    c0 = m0 + lam * rn
    c1 = m1 + lam * rp
    c2 = m2 + lam * rs
    if c0 <= c1 and c0 <= c2: return 0, (m0, m1, m2)
    if c1 <= c2:              return 1, (m0, m1, m2)
    return 2, (m0, m1, m2)


def bits_to_winners(sel_none, sel_pc, sel_ns, n_ru):
    out = []
    for i in range(n_ru):
        if   (sel_none >> i) & 1: out.append(0)
        elif (sel_pc   >> i) & 1: out.append(1)
        elif (sel_ns   >> i) & 1: out.append(2)
        else:                     out.append(-1)   # RU never processed
    return out


# ============================================================================
# NS trainer -- inlined so the notebook is standalone (same math as ns_golden.py)
# ============================================================================
_PATCH, _HALF, _FSZ = 7, 3, 49

def _ns_mask_and_basis():
    m = np.zeros((7, 7), dtype=bool)
    for r, cs in [(0,[3]),(1,[2,3,4]),(2,[1,2,3,4,5]),(3,[1,2,3,4,5]),
                  (4,[1,2,3,4,5]),(5,[2,3,4]),(6,[3])]:
        m[r, cs] = True
    m  = m.ravel()
    mi = np.where(m)[0]
    pos = {int(i): k for k, i in enumerate(mi)}
    seen, cols = set(), []
    for i in mi:
        i = int(i)
        if i in seen: continue
        j = m.size - 1 - i
        c = np.zeros(mi.size, np.float32); c[pos[i]] = 1.0
        if j != i: c[pos[j]] = 1.0
        cols.append(c); seen.update([i, j])
    return m, np.stack(cols, axis=1)

_NS_MASK, _NS_BASIS = _ns_mask_and_basis()
_NS_K     = _NS_BASIS.shape[1]
_NS_WVEC  = _NS_BASIS.sum(0).astype(np.float64)
_NS_REP   = np.array([int(np.argmax(_NS_BASIS[:, k])) for k in range(_NS_K)])


def _train_one_ru(sr_ru, hr_ru, sr_slab):
    win     = sliding_window_view(sr_slab, (_PATCH, _PATCH))
    Xh, Xw  = sr_ru.shape
    patches = win[:Xh, :Xw].reshape(-1, _FSZ)
    Xm      = patches[:, _NS_MASK].astype(np.float64) / 255.0
    y       = hr_ru.astype(np.float64).ravel() / 255.0
    XB      = Xm @ _NS_BASIS
    A       = XB.T @ XB + 0.01 * np.eye(_NS_K)
    b       = XB.T @ y
    KKT = np.zeros((_NS_K + 1, _NS_K + 1))
    KKT[:_NS_K, :_NS_K] = A
    KKT[:_NS_K, _NS_K]  = _NS_WVEC
    KKT[_NS_K, :_NS_K]  = _NS_WVEC
    try:
        f_u = np.linalg.solve(KKT, np.concatenate([b, [1.0]]))[:_NS_K]
    except np.linalg.LinAlgError:
        return np.array([0]*11 + [32], np.int8), 2048
    f_c   = _NS_BASIS @ f_u
    yy    = float(y @ y); N = len(y)
    A_K   = XB.T @ XB; b_K = XB.T @ y
    f_uc  = f_c[_NS_REP].astype(np.float64)
    scale = 63 / max(abs(f_uc).max(), 1e-12)
    f_int = np.clip(np.round(f_uc * scale), -63, 63).astype(np.int64)
    def mse(fi):
        fr = fi / scale
        s  = float(_NS_WVEC @ fr)
        if abs(s) < 1e-6: return float('inf')
        fs = fr / s
        return float((yy - 2 * fs @ b_K + fs @ A_K @ fs) / N)
    best = mse(f_int)
    for _ in range(20):
        improved = False
        for k in range(_NS_K):
            bd = 0
            for d in (-1, 1):
                t = f_int.copy()
                t[k] = np.clip(t[k] + d, -63, 63)
                if t[k] == f_int[k]: continue
                m = mse(t)
                if m < best - 1e-12:
                    best, bd = m, d
            if bd:
                f_int[k] = np.clip(f_int[k] + bd, -63, 63)
                improved = True
        if not improved: break
    ws   = int(2 * f_int[:11].sum() + f_int[11])
    norm = int(round(65536 / ws)) & 0xFFFF if ws != 0 else 0
    return f_int.astype(np.int8), norm


def train_ns_taps(prelr, hr):
    H, W    = prelr.shape
    RU_ROWS = (H + 255) // 256
    RU_COLS = (W + 255) // 256
    sr_pad  = np.pad(prelr, _HALF, mode='symmetric')
    banks   = [(np.zeros(12, np.int8), 0)
               for _ in range((RU_ROWS - 1) * 8 + (RU_COLS - 1) + 1)]
    for r in range(RU_ROWS):
        r0, r1 = r * 256, min((r + 1) * 256, H)
        for c in range(RU_COLS):
            c0, c1 = c * 256, min((c + 1) * 256, W)
            slab   = sr_pad[r0:r1 + 2*_HALF, c0:c1 + 2*_HALF]
            banks[r * 8 + c] = _train_one_ru(
                prelr[r0:r1, c0:c1], hr[r0:r1, c0:c1], slab)
    return banks


# ============================================================================
# HW filter runners -- START pulse BEFORE sendchannel.transfer (pynq_dma_ordering)
# ============================================================================
def _reset_dma(dma):
    mm = dma.mmio
    mm.write(0x00, 0x4); mm.write(0x30, 0x4); time.sleep(0.01)
    mm.write(0x00, 0x1); mm.write(0x30, 0x1)
    dma.sendchannel._first_transfer = True
    dma.recvchannel._first_transfer = True


def run_pc_hw(prelr):
    H, W = prelr.shape
    data = np.load(PC_NPZ)
    T_Q214 = np.round(data['T'] * 16384).astype(int)
    for i, t in enumerate(T_Q214):
        pc_flt.write(i * 4, int(t))
    pc_flt.write(0x1C, (H << 16) | W)

    n_in, n_out = H * W, (H - 6) * (W - 6)
    IN_BUF[:n_in] = prelr.flatten().astype(np.uint32)
    _reset_dma(dma_pcf)
    dma_pcf.recvchannel.transfer(OUT_PC_BUF[:n_out])
    dma_pcf.sendchannel.transfer(IN_BUF[:n_in])
    dma_pcf.sendchannel.wait()
    dma_pcf.recvchannel.wait()
    interior = (OUT_PC_BUF[:n_out] & 0xFF).astype(np.uint8).reshape(H - 6, W - 6)
    out = prelr.copy()
    out[3:H - 3, 3:W - 3] = interior
    return out


def _load_ns_banks(banks, RU_ROWS, RU_COLS):
    for r in range(RU_ROWS):
        for c in range(RU_COLS):
            bank = r * 8 + c
            taps, norm = banks[bank]
            ns_flt.write(0x3C, bank << 8)               # select inactive bank
            for i in range(12):
                ns_flt.write(i * 4, int(taps[i]) & 0x7F)
            ns_flt.write(0x30, int(norm) & 0xFFFF)


def run_ns_hw(prelr, banks):
    H, W = prelr.shape
    RU_ROWS = (H + 255) // 256
    RU_COLS = (W + 255) // 256
    _load_ns_banks(banks, RU_ROWS, RU_COLS)
    ns_flt.write(0x3C, 0x2)                             # FLIP -> banks active
    ns_flt.write(0x34, W); ns_flt.write(0x38, H)

    n_in, n_out = H * W, (H - 6) * W
    IN_BUF[:n_in] = prelr.flatten().astype(np.uint32)
    _reset_dma(dma_nsf)
    dma_nsf.recvchannel.transfer(OUT_NS_BUF[:n_out])
    ns_flt.write(0x3C, 0x1)                             # start pulse BEFORE send
    dma_nsf.sendchannel.transfer(IN_BUF[:n_in])
    dma_nsf.sendchannel.wait()
    dma_nsf.recvchannel.wait()

    # NS output is flat, raster starts at (3, 3). Length = (H-6)*W.
    hw_flat = (OUT_NS_BUF[:n_out] & 0xFF).astype(np.uint8)
    out = prelr.copy()
    view = out.reshape(-1)
    start = 3 * W + 3
    view[start:start + hw_flat.size] = hw_flat
    return out


# ============================================================================
# RDO dispatch -- per-RU 4-way DMA
# ============================================================================
def _extract_ru_flat(frame, ru_row, ru_col):
    r0, c0 = ru_row * RU_SIZE, ru_col * RU_SIZE
    return frame[r0:r0 + RU_SIZE, c0:c0 + RU_SIZE].flatten().astype(np.uint32)


def run_rdo_frame(hr, prelr, pc_full, ns_full, RU_ROWS, RU_COLS):
    n_ru = RU_ROWS * RU_COLS
    rdo.write(CTRL, CTRL_CLEAR); rdo.write(CTRL, 0)
    rdo.write(CTRL, CTRL_SOF);   rdo.write(CTRL, 0)

    sw_wins, mses = [], []
    for ru_idx in range(n_ru):
        r, c = ru_idx // RU_COLS, ru_idx % RU_COLS
        hr_t = _extract_ru_flat(hr,      r, c)
        no_t = _extract_ru_flat(prelr,   r, c)
        pc_t = _extract_ru_flat(pc_full, r, c)
        ns_t = _extract_ru_flat(ns_full, r, c)
        w, m = sw_select_tile(hr_t, no_t, pc_t, ns_t)
        sw_wins.append(w); mses.append(m)

        RU_HR[:] = hr_t; RU_NO[:] = no_t; RU_PC[:] = pc_t; RU_NS[:] = ns_t
        dma_hr.sendchannel.transfer(RU_HR)
        dma_no.sendchannel.transfer(RU_NO)
        dma_pc.sendchannel.transfer(RU_PC)
        dma_ns.sendchannel.transfer(RU_NS)
        for d in [dma_hr, dma_no, dma_pc, dma_ns]:
            d.sendchannel.wait()

    sel_none = (rdo.read(SEL_NONE_HI) << 32) | rdo.read(SEL_NONE_LO)
    sel_pc   = (rdo.read(SEL_PC_HI)   << 32) | rdo.read(SEL_PC_LO)
    sel_ns   = (rdo.read(SEL_NS_HI)   << 32) | rdo.read(SEL_NS_LO)
    hw_wins  = bits_to_winners(sel_none, sel_pc, sel_ns, n_ru)
    return dict(hw_wins=hw_wins, sw_wins=sw_wins, mses=mses,
                status=rdo.read(STATUS))


# ============================================================================
# PSNR + blend for reporting
# ============================================================================
def psnr(a, b):
    m = float(np.mean((a.astype(np.int32) - b.astype(np.int32)) ** 2))
    return float('inf') if m == 0 else 10.0 * np.log10(255.0 ** 2 / m)


def blend_frame(prelr, pc_full, ns_full, winners, RU_ROWS, RU_COLS):
    out = prelr.copy()
    for i, w in enumerate(winners):
        r, c   = i // RU_COLS, i % RU_COLS
        r0, c0 = r * RU_SIZE, c * RU_SIZE
        src = [prelr, pc_full, ns_full][w if w >= 0 else 0]
        out[r0:r0 + RU_SIZE, c0:c0 + RU_SIZE] = src[r0:r0 + RU_SIZE, c0:c0 + RU_SIZE]
    return out


## Main validation loop

Runs every sequence end-to-end. Fast filter runs (~150 ms each) plus one
NS training pass (~20 s per frame). Total ~5 minutes for all 13 sequences.


In [ ]:
results  = []
skipped  = []

for name in sequences:
    if name not in RESOLUTIONS:
        skipped.append((name, 'unknown resolution')); continue
    W, H = RESOLUTIONS[name]
    seq_dir = os.path.join(DATASETS, name)

    for qp_want in QPS:
        prelr_path, hr_path, rng, frame_id, qp = find_representative(seq_dir, qp=qp_want)
        if prelr_path is None:
            skipped.append((name, f'no pre-LR / HR pair for qp={qp_want}')); continue

        prelr = load_raw_y(prelr_path, W, H)
        hr    = load_raw_y(hr_path,    W, H)

        RU_ROWS = H // RU_SIZE          # full RUs only
        RU_COLS = W // RU_SIZE
        n_ru    = RU_ROWS * RU_COLS

        t0 = time.time(); banks   = train_ns_taps(prelr, hr);   dt_train = time.time() - t0
        t0 = time.time(); pc_full = run_pc_hw(prelr);           dt_pc    = time.time() - t0
        t0 = time.time(); ns_full = run_ns_hw(prelr, banks);    dt_ns    = time.time() - t0
        t0 = time.time()
        out = run_rdo_frame(hr, prelr, pc_full, ns_full, RU_ROWS, RU_COLS)
        dt_rdo = time.time() - t0

        matches   = sum(int(h == s) for h, s in zip(out['hw_wins'], out['sw_wins']))
        agree_pct = 100.0 * matches / n_ru
        hw_dist   = Counter(out['hw_wins'])
        sw_dist   = Counter(out['sw_wins'])

        blend_hw = blend_frame(prelr, pc_full, ns_full, out['hw_wins'], RU_ROWS, RU_COLS)

        psnr_pre   = psnr(prelr,    hr)
        psnr_pc    = psnr(pc_full,  hr)
        psnr_ns    = psnr(ns_full,  hr)
        psnr_blend = psnr(blend_hw, hr)

        results.append(dict(
            name=name, frame=frame_id, qp=qp, rng=rng,
            n_ru=n_ru, matches=matches, agree=agree_pct,
            hw_none=hw_dist.get(0, 0), hw_pc=hw_dist.get(1, 0), hw_ns=hw_dist.get(2, 0),
            sw_none=sw_dist.get(0, 0), sw_pc=sw_dist.get(1, 0), sw_ns=sw_dist.get(2, 0),
            psnr_pre=psnr_pre, psnr_pc=psnr_pc, psnr_ns=psnr_ns, psnr_blend=psnr_blend,
            dt_train=dt_train, dt_pc=dt_pc, dt_ns=dt_ns, dt_rdo=dt_rdo,
        ))
        print(f"{name:38s} {frame_id} qp{qp}  "
              f"train={dt_train:5.1f}s  pc={dt_pc*1000:4.0f}ms  "
              f"ns={dt_ns*1000:4.0f}ms  rdo={dt_rdo*1000:5.0f}ms  "
              f"agree={agree_pct:6.2f}%  "
              f"pre={psnr_pre:5.2f} pc={psnr_pc:5.2f} ns={psnr_ns:5.2f} "
              f"blend={psnr_blend:5.2f}dB (+{psnr_blend - psnr_pre:+.2f})")

        del prelr, hr, pc_full, ns_full, blend_hw, banks
        gc.collect()

print(f"\ndone. {len(results)} sequences ran, {len(skipped)} skipped.")
for s, reason in skipped:
    print(f"  skipped {s}: {reason}")


In [ ]:
def m(xs): return float(np.mean(xs))
def g(xs, base): return m(xs) - m(base)                       # dB gain vs base

agrees      = [r['agree']      for r in results]
psnrs_pre   = [r['psnr_pre']   for r in results]
psnrs_pc    = [r['psnr_pc']    for r in results]
psnrs_ns    = [r['psnr_ns']    for r in results]
psnrs_blend = [r['psnr_blend'] for r in results]

print("=" * 108)
print(f"Runs                       : {len(results)}   ({len(sequences)} seqs x {len(QPS)} QPs)")
print(f"Mean per-RU HW-vs-SW agree : {m(agrees):7.3f}%    (worst {min(agrees):.3f}%)")
print(f"Mean PSNR pre-LR vs HR     : {m(psnrs_pre):6.3f} dB    (baseline, no filtering)")
print(f"Mean PSNR PC-alone         : {m(psnrs_pc):6.3f} dB    ({g(psnrs_pc, psnrs_pre):+.3f} dB vs pre-LR)")
print(f"Mean PSNR NS-alone         : {m(psnrs_ns):6.3f} dB    ({g(psnrs_ns, psnrs_pre):+.3f} dB vs pre-LR)")
print(f"Mean PSNR RDO-Combined     : {m(psnrs_blend):6.3f} dB    ({g(psnrs_blend, psnrs_pre):+.3f} dB vs pre-LR)")
print("=" * 108)

# Per-QP means -- shows how PC/NS/Combined behave across the QP sweep
print(f"\nPer-QP means:")
print(f"  {'QP':>4s}  {'#runs':>6s}  {'agree%':>7s}  "
      f"{'preLR':>7s} {'PC':>7s} {'NS':>7s} {'Combined':>8s}   "
      f"{'dPC':>7s} {'dNS':>7s} {'dCombined':>9s}")
for qp in QPS:
    subset = [r for r in results if r['qp'] == qp]
    if not subset: continue
    q_pre   = [r['psnr_pre']   for r in subset]
    q_pc    = [r['psnr_pc']    for r in subset]
    q_ns    = [r['psnr_ns']    for r in subset]
    q_blend = [r['psnr_blend'] for r in subset]
    q_ag    = [r['agree']      for r in subset]
    print(f"  {qp:>4s}  {len(subset):>6d}  {m(q_ag):7.3f}  "
          f"{m(q_pre):7.3f} {m(q_pc):7.3f} {m(q_ns):7.3f} {m(q_blend):8.3f}   "
          f"{g(q_pc, q_pre):+7.3f} {g(q_ns, q_pre):+7.3f} {g(q_blend, q_pre):+9.3f}")

# Per-(sequence, QP) breakdown
print(f"\n{'Sequence':38s} {'Frame':>5s} {'QP':>4s} "
      f"{'Agree%':>7s}  {'HW N/P/S':>10s}  "
      f"{'preLR':>7s} {'PC':>7s} {'NS':>7s} {'Combined':>8s}   {'dCombined':>9s}")
for r in results:
    hw_np = f"{r['hw_none']}/{r['hw_pc']}/{r['hw_ns']}"
    print(f"{r['name']:38s} {r['frame']:>5s} {r['qp']:>4s} "
          f"{r['agree']:7.3f}  {hw_np:>10s}  "
          f"{r['psnr_pre']:7.3f} {r['psnr_pc']:7.3f} {r['psnr_ns']:7.3f} "
          f"{r['psnr_blend']:8.3f}   {r['psnr_blend']-r['psnr_pre']:+9.3f}")


In [ ]:
import matplotlib.pyplot as plt

labels = [f"{r['name'][:18]} qp{r['qp']}" for r in results]
agrees = [r['agree'] for r in results]

fig, ax = plt.subplots(figsize=(13, 4.8))
x = np.arange(len(labels))
colors = ['#2a9d8f' if a >= 99.9999 else '#e76f51' for a in agrees]
ax.bar(x, agrees, color=colors)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=7)
ax.set_ylabel('per-RU HW vs SW agreement %')
ax.set_ylim(min(90, min(agrees) - 1), 100.5)
ax.axhline(100.0, color='#264653', linestyle='--', linewidth=0.5)
n_pass = sum(a >= 99.9999 for a in agrees)
ax.set_title(f'RDO -- per-(sequence, QP) selection agreement\n'
             f'mean {np.mean(agrees):.3f}%   worst {min(agrees):.3f}%   '
             f'({n_pass} / {len(agrees)} runs at 100%)',
             fontsize=10)
plt.tight_layout(); plt.show()


In [ ]:
# Grouped by QP: for each sequence, show pre-LR / PC / NS / Combined side by side.
# Each QP is a separate subplot -- lets you see NS peak at low QP fall off at high QP.
seq_order = list(dict.fromkeys(r['name'] for r in results))
short = [n[:22] for n in seq_order]

fig, axes = plt.subplots(1, len(QPS), figsize=(5.5 * len(QPS), 4.8), sharey=True)
if len(QPS) == 1: axes = [axes]

for ax, qp in zip(axes, QPS):
    subset = {r['name']: r for r in results if r['qp'] == qp}
    if not subset:
        ax.set_visible(False); continue

    xs        = np.arange(len(seq_order))
    pre_vals  = [subset[n]['psnr_pre']   if n in subset else np.nan for n in seq_order]
    pc_vals   = [subset[n]['psnr_pc']    if n in subset else np.nan for n in seq_order]
    ns_vals   = [subset[n]['psnr_ns']    if n in subset else np.nan for n in seq_order]
    bl_vals   = [subset[n]['psnr_blend'] if n in subset else np.nan for n in seq_order]

    w = 0.2
    ax.bar(xs - 1.5*w, pre_vals, w, color='#94a3b8', label='pre-LR')
    ax.bar(xs - 0.5*w, pc_vals,  w, color='#f4a261', label='PC-alone')
    ax.bar(xs + 0.5*w, ns_vals,  w, color='#2a9d8f', label='NS-alone')
    ax.bar(xs + 1.5*w, bl_vals,  w, color='#264653', label='RDO-Combined')
    ax.set_xticks(xs); ax.set_xticklabels(short, rotation=45, ha='right', fontsize=8)
    dPC = float(np.mean(pc_vals) - np.mean(pre_vals))
    dNS = float(np.mean(ns_vals) - np.mean(pre_vals))
    dBL = float(np.mean(bl_vals) - np.mean(pre_vals))
    ax.set_title(f'QP {qp}\ndPC {dPC:+.2f}   dNS {dNS:+.2f}   dCombined {dBL:+.2f}  dB',
                 fontsize=10)

axes[0].set_ylabel('PSNR vs HR (dB)  --  higher is better')
axes[-1].legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()


## Cleanup

Order matters: free the persistent CMA buffers **before** `ol.free()` so PYNQ
does not walk descriptor rings that reference freed memory
(see `feedback_pynq_cma_buffer_reuse`).


In [ ]:
for b in [IN_BUF, OUT_NS_BUF, OUT_PC_BUF, RU_HR, RU_NO, RU_PC, RU_NS]:
    b.freebuffer()
ol.free()
gc.collect()
print("clean teardown complete")
